In [ ]:
!pip install black
!pip install "black[jupyter]"

In [ ]:
!black "requests_(1) (1).ipynb"

In [ ]:
import requests

In [ ]:
url = "https://hirehi.ru/api/search/jobs?limit=50"
page = requests.get(url)

In [ ]:
page

In [ ]:
print(page.json())

In [ ]:
a = [page.json()["limit"], page.json()["page"], page.json()["total_count"]]
a

In [ ]:
import math

pageamount = math.ceil(a[2] / a[0])
pageamount

In [ ]:
data = page.json()
jobs = data["jobs"]
first = jobs[0]
print(first.keys())

In [ ]:
def get_page(p):
    url = f"https://hirehi.ru/api/search/jobs?page={p}&limit=50"
    response = requests.get(url)
    jobs = response.json()["jobs"]
    clean_data = []
    for i in jobs:
        cleanvac = {
            "id": i.get("id"),
            "title": i.get("title"),
            "category": i.get("category"),
            "company": i.get("company"),
            "format": i.get("format"),
            "level": i.get("level"),
            "salary": i.get("salary"),
        }
        clean_data.append(cleanvac)
    return clean_data

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
infa = []
for p in tqdm(range(1, pageamount + 1)):
    try:
        infa.extend(get_page(p))
        sleep(3)
    except:
        print(p)

In [ ]:
import pandas as pd

df = pd.DataFrame(infa)
print(df.shape)
df.head(10)

Далее соберем ссылки:</br>
.NET Developer</br>
https://hirehi.ru/development/net-developer-36177</br>
графический дизайнер</br>
https://hirehi.ru/design/graficheskii-dizainer-36174</br>
Сылки выглядят таким образом , то есть https://hirehi.ru/{category}/{title}-{id}
</br>Все title приведены к нижнему регистру, текст на русском транслитерирован, все символы, кроме букв заменены на "-", крайние дефисы удалены, для этого можно использовать модуль **transliterate**.
Создадим дополнительные колонки для транслитерации русского текста в колонках **title** и **category**.

In [ ]:
from transliterate import slugify

text = "URL-адреса/http"
text1 = "URL"
link = slugify(text)
link1 = slugify(text1)
print(link)
print(link1)

Если текст состоит из слов только на английском, то выводится none, значит текст, который написан на английском оставляем как есть. </br> Такжде слэши удаляются и не заменяются на дефисы, как должно быть с карточкой UX/UI дизайнер https://hirehi.ru/design/ux-ui-dizainer-36159

In [ ]:
df["title_trans"] = df["title"]
df["category_trans"] = df["category"]
df["id_trans"] = df["id"]
df.head(10)

In [ ]:
df["title_trans"] = df["title_trans"].astype(str).str.lower()
df["category_trans"] = df["category_trans"].astype(str).str.lower()
df["id_trans"] = df["id_trans"].astype(str)
df.head(10)

In [ ]:
df["title_trans"] = (
    df["title_trans"]
    .str.replace(r"[^\w]", "-", regex=True)
    .str.replace(r"_", "-", regex=True)
)
df["category_trans"] = (
    df["category_trans"]
    .str.replace(r"[^\w]", "-", regex=True)
    .str.replace(r"_", "-", regex=True)
)
df

In [ ]:
df[df["id"] == 36177]

нужно удалить повторяющиеся дефисы и дефисы в начале/конце

In [ ]:
df["title_trans"] = df["title_trans"].str.replace(r"-+", "-", regex=True)
df["category_trans"] = df["category_trans"].str.replace(r"-+", "-", regex=True)
df

In [ ]:
df["title_trans"] = df["title_trans"].str.strip("-")
df["category_trans"] = df["category_trans"].str.strip("-")
df

напишем функцию, которая будет транслитерировать русский текст, а английский оставлять как есть


<b>НЕПРАВИЛЬНО РАБОТАЕТ</b>


In [ ]:
from pytils.translit import slugify


def makelink(i):
    ready = slugify(i)
    if ready == None:
        return i
    return ready

<b>ЗАПУСКАТЬ ЭТО</b>

In [ ]:
def makelink1(i):
    letters = {
        "а": "a",
        "б": "b",
        "в": "v",
        "г": "g",
        "д": "d",
        "е": "e",
        "ё": "e",
        "ж": "zh",
        "з": "z",
        "и": "i",
        "й": "i",
        "к": "k",
        "л": "l",
        "м": "m",
        "н": "n",
        "о": "o",
        "п": "p",
        "р": "r",
        "с": "s",
        "т": "t",
        "у": "u",
        "ф": "f",
        "х": "h",
        "ц": "ts",
        "ч": "ch",
        "ш": "sh",
        "щ": "sch",
        "ъ": "",
        "ы": "y",
        "ь": "",
        "э": "e",
        "ю": "yu",
        "я": "ya",
    }
    ready = ""
    for s in i:
        if s in letters:
            ready += letters[s]
        else:
            ready += s
    return ready

In [ ]:
df["title_trans"] = df["title_trans"].apply(makelink1)
df["category_trans"] = df["category_trans"].apply(makelink1)
df

Теперь соберем ссылку в том виде, в котором она должна быть <br>
https://hirehi.ru/{category}/{title}-{id}


In [ ]:
df["link"] = (
    "https://hirehi.ru/"
    + df["category_trans"]
    + "/"
    + df["title_trans"]
    + "-"
    + df["id_trans"]
)
df

In [ ]:
df[df["id"] == 36174]

мы получили ссылки, но можно заметить такую проблему, что transliterate заменяет буквы не так "по-русски", в наших ссылках пишется **ux-ui-dizajner**, когда в оригинальной ссылке используется **ux-ui-dizainer**
значит библиотека не сможет "правильно" транслитерировать наш текст. Поэтому напишем словарь, с помощью которого можно будет просто заменять русскую букву на английскую, без дополнительных правил библиотеки, когда две гласные стоят рядом и происходит удвоение.


In [ ]:
df.drop(columns=["title_trans", "category_trans", "id_trans"], inplace=True)
df

In [ ]:
df.to_csv("vacancys.csv", index=False)